# RNP vs DFST on CIFAR-10

Runs Reconstructive Neuron Pruning (ICML 2023) against a DFST-backdoored ResNet-18.

**Settings → Accelerator: GPU (T4/P100)**. Internet must be ON for the git clone and the CIFAR-10 download.

Replace `REPO_URL` below with your repository link.

In [ ]:
REPO_URL = 'https://github.com/Shuvayu12/DFST_RNP.git'
TARGET_LABEL = 0                                     # DFST target class


In [ ]:
import os, torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
!git clone $REPO_URL RNP
%cd RNP
!ls -la


## 1. Convert the DFST checkpoints

The two `.pt` files are whole pickled models from the original DFST code. `convert_dfst.py` turns them into plain state dicts under `weights/`.

In [ ]:
!mkdir -p weights logs
!python convert_dfst.py --clf cifar10_resnet18_dfst.pt --gen cifar10_resnet18_dfst_generator.pt --out weights/
!ls -la weights/


## 2. Sanity check the backdoor

One normalization row should show clean accuracy around 0.9 **and** ASR around 0.9+. If that row is not the `repo` one, edit `MEAN_CIFAR10 / STD_CIFAR10` in `data/poison_tool_cifar.py` before continuing.

In [ ]:
!python check_dfst.py --n 2000 --target_label $TARGET_LABEL


## 3. Run RNP

Unlearn → recover masks → prune. Same hyper-parameters as the paper's BadNets run; only the trigger type and model path change. Takes roughly 10–15 minutes on a T4.

In [ ]:
!python main.py \n    --arch resnet18 \n    --backdoor_model_path weights/dfst_resnet18.tar \n    --trigger_type dfstTrigger \n    --dfst_generator_path weights/dfst_generator.pt \n    --target_label $TARGET_LABEL \n    --target_type all2one \n    --output_weight weights/ \n    --log_root logs/


## 4. Results

`logs/pruning_by_threshold.txt` lists, for each mask threshold, how many neurons were pruned and the resulting attack success rate (PoisonACC) and clean accuracy (CleanACC).

In [ ]:
import pandas as pd
df = pd.read_csv('logs/pruning_by_threshold.txt', sep=r'\s*	\s*', engine='python')
df


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(df['Mask'], df['PoisonACC'], marker='o', label='Attack success rate')
ax.plot(df['Mask'], df['CleanACC'], marker='s', label='Clean accuracy')
ax.set_xlabel('pruning threshold on mask value'); ax.set_ylabel('accuracy'); ax.set_title('RNP on DFST (ResNet-18, CIFAR-10)')
ax.legend(); ax.grid(alpha=.3); plt.show()


## 5. Keep the outputs

Copy logs and the unlearned checkpoint to `/kaggle/working` so they survive *Save & Run All*.

In [ ]:
!mkdir -p /kaggle/working/rnp_dfst_out
!cp -r logs /kaggle/working/rnp_dfst_out/
!cp weights/unlearned_model_last.tar /kaggle/working/rnp_dfst_out/ 2>/dev/null || true
!ls -la /kaggle/working/rnp_dfst_out
